# 320 caminhos para uma mesma carteira

Exact GBM, reproducible statistics, and black-chart exports. Este notebook usa premissas ilustrativas; **não é recomendação de investimento**.

**Repo privado / Colab:** baixe este `.ipynb` e o ZIP do código pelo GitHub autenticado. Abra o notebook em **File → Upload notebook** no Colab; execute a primeira célula e envie o ZIP quando solicitado. Nenhum token é usado. Depois da publicação, o badge do README abrirá o notebook diretamente.

Localmente, instale o pacote com `pip install -e .` e abra este arquivo no Jupyter.


In [ ]:
# Instalação sem credenciais: pacote local, ou ZIP enviado à sessão Colab.
import importlib.util
import io
import subprocess
import sys
import zipfile
from pathlib import Path

if importlib.util.find_spec("portfolio_sim") is None:
    if "google.colab" in sys.modules:
        from google.colab import files

        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Envie exatamente um ZIP do código-fonte deste repo.")
        destination = Path("/content/portfolio-source").resolve()
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
            for entry in archive.infolist():
                if not (destination / entry.filename).resolve().is_relative_to(destination):
                    raise ValueError("ZIP contém um caminho fora da pasta de destino.")
            archive.extractall(destination)
        projects = list(destination.glob("*/pyproject.toml"))
        if len(projects) != 1:
            raise ValueError("Use o ZIP do GitHub, com uma pasta contendo pyproject.toml.")
        project = projects[0].parent
    else:
        projects = [
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "src/portfolio_sim").is_dir()
        ]
        if not projects:
            raise RuntimeError("Instale o pacote com pip install -e . na raiz do repo.")
        project = projects[0]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(project)])

from portfolio_sim.model import Config, analytical_statistics, simulate
from portfolio_sim.render import export_animation, export_png

print("Pacote pronto. Nenhuma credencial foi usada.")

## Premissas e solução analítica

`annual_return` é o drift μ do GBM. Com μ = 0.08, o crescimento esperado anual é `exp(0.08)−1`, cerca de 8.33%. O valor mediano cresce a μ−σ²/2. O modelo não inclui aportes, custos, impostos ou inflação.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Image, Video, display

config = Config(start=100_000, years=10, annual_return=0.08, volatility=0.17, paths=320, seed=42)
result = simulate(config)
assert (result.values > 0).all()
np.testing.assert_array_equal(result.values, simulate(config).values)
comparison = pd.DataFrame(
    {"320 caminhos": result.statistics, "analítico": analytical_statistics(config)}
)
display(comparison)

## O gráfico

O eixo Y é logarítmico: distâncias iguais são multiplicadores iguais. A distribuição usa bins geométricos e contagens, com a mesma escala. A cor usa o valor **terminal**, informação retrospectiva. A linha branca é a mediana; a probabilidade de perda compara apenas valor final e inicial. Nenhum outlier é cortado.


In [ ]:
output = Path("output/notebook")
for layout in ("wide", "vertical"):
    path = export_png(result, output / f"simulation-{layout}.png", layout=layout)
    display(Image(filename=str(path), width=1000 if layout == "wide" else 420))

## Quanto os 320 caminhos erram?

A amostra do gráfico não é uma estimativa precisa das caudas. Aumente o número de caminhos para comparar com a distribuição lognormal exata; não é necessário renderizar todos eles.


In [ ]:
larger = simulate(Config(paths=50_000, seed=817))
expected = analytical_statistics(larger.config)
for key in ("mean", "median", "p5", "p95"):
    assert abs(larger.statistics[key] / expected[key] - 1) < 0.025
assert abs(larger.statistics["loss_probability"] - expected["loss_probability"]) < 0.006
display(pd.DataFrame({"50 mil caminhos": larger.statistics, "analítico": expected}))
del larger

## Calibração offline, com origem explícita

A fixture é **sintética**, criada em 06/11/2026 para testar o adaptador; não são preços de WEGE3 nem do IBOV. A calibração usa σ = std(log-retornos)×√252 e μ = mean(log-retornos)×252 + σ²/2.

Para dados reais, instale `pip install -e '.[market]'` no repo e rode `portfolio-sim --ticker WEGE3` ou `portfolio-sim --ibov`. O comando registra fonte e datas; não substitui falhas de rede por dados falsos. Documentação do adaptador: https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html


In [ ]:
from portfolio_sim.market import load_prices

calibration = load_prices("WEGE3", offline=True)
assert calibration.provenance["kind"] == "synthetic_fixture"
display(calibration.to_dict())

## MP4 e GIF (opcional, 8 segundos / 15 fps)

Mude `RENDER_VIDEO` para `True` para gerar os dois formatos, em 16:9 e 9:16. ffmpeg precisa estar instalado (`brew install ffmpeg` ou `sudo apt-get install ffmpeg`). Os gráficos e a calibração acima funcionam sem ele.


In [ ]:
import shutil

RENDER_VIDEO = False
if RENDER_VIDEO:
    if not shutil.which("ffmpeg"):
        raise RuntimeError("Instale ffmpeg para exportar MP4 e GIF.")
    for layout in ("wide", "vertical"):
        for extension in ("mp4", "gif"):
            export_animation(
                result,
                output / f"simulation-{layout}.{extension}",
                layout=layout,
                seconds=8,
                fps=15,
            )
    display(Video(str(output / "simulation-wide.mp4"), embed=True))
else:
    print("PNG pronto. Ative RENDER_VIDEO para codificar os vídeos.")